In [ ]:
import os
os.chdir(r"D:\doc\2026\FINAL_CODE\src")

D:\doc\2026\FINAL_CODE\src


In [3]:
dir_save_path="D:/doc/2026/FINAL_CODE/src/codeStat/modelling/voiceModelling/srcVoiceModelling/GMM_TDNN/nonfix_out/GMM"

In [4]:
# Load CSV
df_csv = pd.read_csv("D:/doc/2026/FINAL_CODE/src/codeStat/preModelling/samplingStratified/out_samplingStratified/dataSampledSmall.csv")

# Buat dictionary
dict_transkrip = {
    os.path.splitext(os.path.basename(path))[0]: kalimat
    for path, kalimat in zip(df_csv["Path_Data"], df_csv["sentences"])
}

def baca_transkrip(wav_path: str) -> str:
    audio_id = os.path.splitext(os.path.basename(wav_path))[0]
    return dict_transkrip.get(audio_id, "").lower()

In [ ]:
SAMPLED_ROOT = r"D:\doc\2026\FINAL_CODE\data\sampled_small"

CSV_PATH = (
    r"D:\doc\2026\FINAL_CODE\src\codeStat\preModelling"
    r"\samplingStratified\out_samplingStratified\dataSampledSmall.csv"
)

DIR_SAVE = (
    r"D:\doc\2026\FINAL_CODE\src\codeStat\modelling"
    r"\voiceModelling\srcVoiceModelling\GMM_TDNN\out\GMM"
)

In [ ]:
N_COMPONENTS     = 16     # K komponen GMM per kata
N_ITER           = 100    # maks iterasi EM
TOL              = 1e-4   # toleransi konvergensi
N_GRAM           = 3      # orde N-gram (trigram)
TOP_K_DECODE     = 10     # kandidat kata untuk N-gram rescore
JALANKAN_TUNING  = True   # jalankan hyperparameter tuning (bisa dimatikan)



In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MAIN — Entry Point
# ══════════════════════════════════════════════════════════════════════════════

def main():
    os.makedirs(DIR_SAVE, exist_ok=True)
    t_start = time.time()

    print("=" * 60)
    print("  GMM ASR BAHASA JAWA — Pipeline Efisien (All-in-One)")
    print("=" * 60)

    # [1] Load transkrip
    print("\n[1] Membaca transkrip CSV ...")
    dict_transkrip  = baca_transkrip_csv(CSV_PATH)
    semua_transkrip = list(dict_transkrip.values())
    print(f"    Total audio: {len(dict_transkrip)}")

    lexicon = buat_lexicon(semua_transkrip)
    pd.DataFrame([
        {"Kata": k, "Fonem": " ".join(v)} for k, v in lexicon.items()
    ]).to_csv(f"{DIR_SAVE}/HasilLexicon.csv", index=False)
    print(f"    Vocab: {len(lexicon)} kata unik  →  HasilLexicon.csv ✓")

    # [2] Extract MFCC → cache (SEKALI SAJA)
    print("\n[2] Ekstraksi MFCC ke cache ...")
    cache_mfcc, meta_list = build_mfcc_cache(SAMPLED_ROOT, dict_transkrip)
    pd.DataFrame(meta_list).to_csv(f"{DIR_SAVE}/HasilMFCC.csv", index=False)
    print(f"    HasilMFCC.csv ✓  ({len(cache_mfcc)} audio di-cache)")

    if not cache_mfcc:
        print("  [ERROR] Tidak ada audio berhasil di-load. Cek SAMPLED_ROOT.")
        return

    # [3] Kumpulkan frame per kata (dari cache)
    print("\n[3] Mengumpulkan frame per kata ...")
    frame_per_kata = kumpulkan_frame_per_kata(cache_mfcc, dict_transkrip)

    # [4] GMM Global (untuk referensi)
    print("\n[4] Melatih GMM Global ...")
    X_global   = np.vstack(list(cache_mfcc.values()))
    print(f"    Shape X_global: {X_global.shape}")
    gmm_global = GaussianMixtureModel(
        n_components=N_COMPONENTS, n_iter=N_ITER, tol=TOL,
        reg_covar=1e-6, random_state=42
    )
    gmm_global.fit(X_global)
    pd.DataFrame([{
        "K Komponen"          : N_COMPONENTS,
        "Iterasi EM"          : gmm_global.n_iter_done_,
        "Konvergen"           : gmm_global.converged_,
        "Log-Likelihood Akhir": round(gmm_global.log_likelihood_history_[-1], 6),
    }]).to_csv(f"{DIR_SAVE}/HasilGMMGlobal.csv", index=False)
    print(f"    Konvergen={gmm_global.converged_}  "
          f"Iter={gmm_global.n_iter_done_}  "
          f"LL={gmm_global.log_likelihood_history_[-1]:.4f}  "
          f"→ HasilGMMGlobal.csv ✓")

    # [5] GMM per kata
    print(f"\n[5] Melatih GMM per kata (K={N_COMPONENTS}) ...")
    gmm_per_kata = latih_gmm_per_kata(
        frame_per_kata, n_components=N_COMPONENTS,
        n_iter=N_ITER, tol=TOL
    )
    hasil_gmm_pk = []
    for kata, gmm in gmm_per_kata.items():
        hasil_gmm_pk.append({
            "Kata"                : kata,
            "Jumlah Frame"        : np.vstack(frame_per_kata[kata]).shape[0],
            "K Komponen"          : gmm.n_components,
            "Iterasi EM"          : gmm.n_iter_done_,
            "Konvergen"           : gmm.converged_,
            "Log-Likelihood Akhir": round(gmm.log_likelihood_history_[-1], 4),
        })
    pd.DataFrame(hasil_gmm_pk).to_csv(f"{DIR_SAVE}/HasilGMMPerKata.csv", index=False)
    print(f"    HasilGMMPerKata.csv ✓  ({len(hasil_gmm_pk)} kata)")

    # [6] N-Gram Language Model
    print(f"\n[6] Melatih {N_GRAM}-Gram Language Model ...")
    ngram_model = NGramModel(n=N_GRAM, discount=0.75)
    ngram_model.fit(semua_transkrip)
    print(f"    Vocab={len(ngram_model.vocab)}  N-gram={len(ngram_model.ngram_count)}")

    # [7] Dekode + WER
    print("\n[7] Dekode audio + hitung WER ...")
    hasil_wer = dekode_semua_audio(cache_mfcc, dict_transkrip, gmm_per_kata, ngram_model)
    df_wer    = pd.DataFrame(hasil_wer)
    df_wer.to_csv(f"{DIR_SAVE}/HasilWER_GMM.csv", index=False)
    print(f"    HasilWER_GMM.csv ✓  ({len(df_wer)} audio)")

    stat = ringkasan_wer(df_wer, label="GMM per Kata")
    pd.DataFrame([stat]).to_csv(f"{DIR_SAVE}/HasilRingkasanWER.csv", index=False)

    # [8] Hyperparameter Tuning
    if JALANKAN_TUNING:
        print("\n[8] Hyperparameter Tuning ...")
        hyperparameter_tuning(
            frame_per_kata, cache_mfcc, dict_transkrip,
            ngram_model, DIR_SAVE
        )
    else:
        print("\n[8] Tuning dilewati (JALANKAN_TUNING=False)")

    # Selesai
    total = time.time() - t_start
    print("\n" + "=" * 60)
    print(f"  SELESAI dalam {total/60:.1f} menit")
    print(f"  Output: {DIR_SAVE}")
    print("=" * 60)
    for f in ["HasilLexicon", "HasilMFCC", "HasilGMMGlobal",
              "HasilGMMPerKata", "HasilWER_GMM", "HasilRingkasanWER",
              "HasilHyperparameterTuning_GMM"]:
        path = f"{DIR_SAVE}/{f}.csv"
        status = "✓" if os.path.isfile(path) else "–"
        print(f"  {status}  {f}.csv")


if __name__ == "__main__":
    main()